# Non-Local LLM Inference

In [ ]:
# import packages used in this script
# built-in packages
import os
import textwrap
import time
import re

# pypi packages
import torch
import pandas as pd
from huggingface_hub import login
import outlines
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from pydantic import BaseModel
from enum import Enum


# own packages
import start

# Local Inference
Local calls are another way to interact with large language models in Python. 

We download the model from HuggingFace and run prompts against it locally. Our hardware specs become a consideration in this case.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\nUsing device: {DEVICE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

Colab Option A) Save API key to Secrets

Navigate to Secrets in the left-hand panel, create a variable called `HF_TOKEN`. 

ADD SCREENSHOT


Local Option B) Save API key to .env


- Add your keys to `.env`:
   ```
   HF_TOKEN = ""
   ```

In [ ]:
# check that the api key loaded to the environment
# if loaded, prints the key
# if not loaded, prints ERROR
key = "HF_TOKEN"
print(os.environ.get(key, f"ERROR: Variable {key} Not Found"))

## ⚙️ Model Settings

We'll use meta-llama/Llama-3.2-1B-Instruct. Here's a link to information about this model:
- https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

Note: The models ending in "Instruct" have been instruction-tuned.

In [ ]:
MODEL = "meta-llama/Llama-3.2-1B-Instruct" 

**Task**



**Temperature**

A parameter (ranging from 0-2) that controls the randomness of text generated by LLMs during inference. Each token is assigned a probability of occurance based on the prompt and the tokens that come before it. Temperature modifies this probability distribution such that at higher temperatures increase the likelihood of selecting less probable tokens.

**Precision**

The number of bits used to store the model's weights and how they're allocated. 
- FP32 (32-bit floating point) — "full precision." High accuracy, but uses the most memory and compute.
- FP16 (16-bit floating point) — "half precision." Much smaller memory footprint, faster on modern GPUs, but a narrower numeric range (can cause overflow/underflow issues).
- BF16 (bfloat16) — also 16-bit, but allocates bits differently than FP16.

For a complete list see:




In [ ]:
TASK = "text-generation"
TOKENS = 500
TEMPERATURE = 0.1
PRECISION = torch.bfloat16 # can use bfloat16 or bfloat32 if cuda is available (float for cpu, bfloat for gpu)

In [ ]:
# initialize the model
model = AutoModelForCausalLM.from_pretrained(MODEL, 
                                            dtype = PRECISION,
                                            token = os.getenv("HF_TOKEN"),
                                            device_map = DEVICE)

hf_tokenizer = AutoTokenizer.from_pretrained(MODEL, token = os.getenv("HF_TOKEN"))

client = outlines.from_transformers(model, hf_tokenizer)

## Prompt llama-3.2-1B

In [ ]:
PROMPT = "What is a dialogic prompt?"

response = client(PROMPT, max_new_tokens = TOKENS)

print(response)

### Classify a Dialogic Prompt

In [ ]:
CASE = "Okay, and what would you do?"

PROMPT = f"""Classify the following utterance as yes or no based on whether it is a dialogic prompt. 
A dialogic prompt is defined as an utterance that implies, encourages, requests, or expects a 
new speaker (or multiple new speakers) to make a verbal contribution.\n
Here is the utterance: 
\n\"\"\"{CASE}\"\"\"\n
Output format:\n
- First, give a brief explanation (1 short sentence, under 20 words) justifying your answer.
- Then give your final answer as a yes or no surrounded by three backticks. For example, ```yes``` or ```no```"""

print(textwrap.fill(PROMPT, width = 92))

In [ ]:
response = client(PROMPT, max_new_tokens = TOKENS)

print(response)

### Classify a Dialogic Prompt with the LLM Codebook

In [ ]:
# load the prompt codebook
path_to_prompts = start.CODEBOOK_DIR / "llm_prompt_codebook.xlsx"
prompts = pd.read_excel(path_to_prompts)

In [ ]:
CASE = "Okay, and what would you do?"

In [ ]:
PROMPT = (prompts.loc[prompts.id == "Coding2", "prompt"].item() + " " +
          prompts.loc[prompts.id == "Construct", "prompt"].item() + " " +
          prompts.loc[prompts.id == "Prompt1", "prompt"].item() + " " +
          f"\n\"\"\"{CASE}\"\"\"\n" + 
          prompts.loc[prompts.id == "Format2", "prompt"].item()
          )

print(PROMPT)

In [ ]:
response = client(PROMPT, max_new_tokens = TOKENS)

print(response)

### Classify Many Dialogic Prompts with the LLM Codebook

In [ ]:
# import data
DATA_SOURCE = "train"

path_to_data = start.DATA_DIR / f"{DATA_SOURCE}.xlsx"
df = pd.read_excel(path_to_data)

In [ ]:
SAMPLE_SIZE = 5

sample = df.sample(n = SAMPLE_SIZE, ignore_index = True)

In [ ]:
for row in sample.index:
    print(row)

In [ ]:
for row in sample.index:
    CASE = sample.loc[row, "text"]

    # construct prompt w case
    PROMPT = (prompts.loc[prompts.id == "Coding2", "prompt"].item() + " " +
              prompts.loc[prompts.id == "Construct", "prompt"].item() + " " +
              prompts.loc[prompts.id == "Prompt1", "prompt"].item() + " " +
              f"\n\"\"\"{CASE}\"\"\"\n" + 
              prompts.loc[prompts.id == "Format2", "prompt"].item()
              )

    response = client(PROMPT, max_new_tokens = TOKENS)
    
    print(f"Utterance: \"{CASE}\"")
    print(f"Response: {response}\n")

#### 📋 Formatting Output
Force the model to respond in a specific format. In this case we want to separate the explanation from the classification (yes or no)

In [ ]:
# format output
class Answer(str, Enum):
    yes = "yes"
    no = "no"

class Classification(BaseModel):
    explanation: str  # comes first, so reasoning happens before the label
    label: Answer

label_map = {"yes": 1, "no": 0}

In [ ]:
# initialize output objects
code_local = []
explanation_local = []

tp = 0
tn = 0
fp = 0
fn = 0
format_errors = 0

# format response
pattern = r"```(yes|no)```"
label_map = {"yes": 1, "no": 0}

In [ ]:
start_time = time.perf_counter()

In [ ]:
for row in sample.index:
    CASE = sample.loc[row, "text"]

    # construct prompt w case
    PROMPT = (prompts.loc[prompts.id == "Coding2", "prompt"].item() + " " +
              prompts.loc[prompts.id == "Construct", "prompt"].item() + " " +
              prompts.loc[prompts.id == "Prompt1", "prompt"].item() + " " +
              f"\n\"\"\"{CASE}\"\"\"\n" + 
              prompts.loc[prompts.id == "Format2", "prompt"].item()
              )

    response = client(PROMPT, Classification, max_new_tokens = TOKENS)

    try:
        parsed = Classification.model_validate_json(response)
        response_code = label_map[parsed.label.value]
        explanation_text = parsed.explanation
    except Exception as e:
        print(f"Row {row} failed: {e}")
        response_code = None
        explanation_text = response
        format_errors = format_errors + 1

    code_local.append(response_code)
    explanation_local.append(explanation_text)

    if sample.loc[row, "code_human"] == 1 and response_code == 1:
        tp = tp + 1
    elif sample.loc[row, "code_human"] == 0 and response_code == 0:
        tn = tn + 1
    elif sample.loc[row, "code_human"] == 0 and response_code == 1:
        fp = fp + 1
    elif sample.loc[row, "code_human"] == 1 and response_code == 0:
        fn = fn + 1

    print(f"Utterance: \"{CASE}\"")
    print(f"LLM Response: {response_code}")
    print(f"Human Response: {sample.loc[row, "code_human"]}\n")

end_time = time.perf_counter()

In [ ]:
sample[f"code_{MODEL}"] = code_local
sample[f"explanation_{MODEL}"] = explanation_local

In [ ]:
if "/" in MODEL:
    model_stripped = re.split("/", MODEL)[1]
else:
    model_stripped = MODEL

In [ ]:
# ---- save the results ----
# classifications
results_data_file = f"{DATA_SOURCE}_{model_stripped}.xlsx"
path_to_data_results = start.RESULTS_DIR / "local" / results_data_file

sample.to_excel(path_to_data_results, index = False)

# model performance
path_to_model_results = start.RESULTS_DIR / "classifications.xlsx"
results = pd.read_excel(path_to_model_results, sheet_name = f"{DATA_SOURCE}")

new_row = {"model": MODEL,
           "utterances": sample.shape[0],
           "tp": tp,
           "tn": tn,
           "fp": fp,
           "fn": fn,
           "runtime": end_time - start_time
           }

results = pd.concat([results, pd.DataFrame([new_row])], ignore_index = True)

with pd.ExcelWriter(
    path_to_model_results, engine = "openpyxl", mode = "a", if_sheet_exists = "replace") as writer:
    results.to_excel(writer, sheet_name=f"{DATA_SOURCE}", index = False)